In [0]:
import mlflow
import mlflow.spark
from pyspark.sql.functions import col, when, datediff, lit, log1p, round as spark_round
from pyspark.ml.feature import VectorAssembler, StringIndexer, StandardScaler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator
)
from pyspark.ml import Pipeline

In [0]:
gold = spark.read.table("bi.gold.customer_features_ml")
gold.printSchema()
display(gold)

In [0]:
from pyspark.sql.functions import col, when, max as spark_max, datediff

max_order_date = gold.agg(spark_max("last_order_date")).collect()[0][0]

print("Ostatnia data w danych:", max_order_date)

gold_fixed = gold.withColumn(
    "days_since_last_order_fixed",
    datediff(lit(max_order_date), col("last_order_date"))
)

PREDICTION_WINDOW = 30

df = gold_fixed.withColumn(
    "will_buy_again",
    when(col("days_since_last_order_fixed") <= PREDICTION_WINDOW, 1).otherwise(0)
)

df.groupBy("will_buy_again").count().show()

In [0]:
numeric_features = [
    "total_revenue",
    "total_orders",
    "avg_order_value",
    "unique_products",
    "unique_categories",
    "avg_freight",
    "customer_lifetime_days",
    "estimated_clv",

]

In [0]:
country_indexer = StringIndexer(
    inputCol="customer_country",
    outputCol="country_idx",
    handleInvalid="keep"
)

segment_indexer = StringIndexer(
    inputCol="customer_segment",
    outputCol="segment_idx",
    handleInvalid="keep"
)

all_features = numeric_features + ["country_idx", "segment_idx"]

assembler = VectorAssembler(
    inputCols=all_features,
    outputCol="features",
    handleInvalid="keep"
)

scaler = StandardScaler(
    inputCol="features",
    outputCol="scaled_features",
    withStd=True,
    withMean=True
)


In [0]:
train_df, test_df = df.randomSplit([0.75, 0.25], seed=42)
print(f"Train: {train_df.count()} | Test: {test_df.count()}")

In [0]:
import pandas as pd
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

pdf = df.select(
    "total_revenue",
    "total_orders",
    "avg_order_value",
    "unique_products",
    "unique_categories",
    "avg_freight",
    "customer_lifetime_days",
    "estimated_clv",
    "customer_country",
    "customer_segment",
    "will_buy_again"
).toPandas()

pdf = pd.get_dummies(pdf, columns=["customer_country", "customer_segment"], drop_first=True)

X = pdf.drop(columns=["will_buy_again"])
y = pdf["will_buy_again"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

with mlflow.start_run(run_name="sklearn_logistic_customer_purchase"):

    clf = LogisticRegression(max_iter=1000)
    clf.fit(X_train_scaled, y_train)

    y_pred = clf.predict(X_test_scaled)
    y_prob = clf.predict_proba(X_test_scaled)[:, 1]

    metrics = {
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "F1": round(f1_score(y_test, y_pred), 4),
        "Precision": round(precision_score(y_test, y_pred), 4),
        "Recall": round(recall_score(y_test, y_pred), 4),
        "AUC": round(roc_auc_score(y_test, y_prob), 4)
    }

    mlflow.log_metrics(metrics)

    print("=== WYNIKI MODELU ===")
    for k, v in metrics.items():
        print(f"{k}: {v}")

In [0]:
base_pdf = df.select(
    "customer_id",
    "customer_name",
    "customer_country",
    "customer_segment",
    "days_since_last_order_fixed",
    "total_orders",
    "total_revenue",
    "estimated_clv",
    "will_buy_again",
    "avg_order_value",
    "unique_products",
    "unique_categories",
    "avg_freight",
    "customer_lifetime_days"
).toPandas()

pred_pdf = base_pdf.copy()

X_all = pred_pdf[
    [
        "total_revenue",
        "total_orders",
        "avg_order_value",
        "unique_products",
        "unique_categories",
        "avg_freight",
        "customer_lifetime_days",
        "estimated_clv",
        "customer_country",
        "customer_segment"
    ]
]

X_all = pd.get_dummies(X_all, columns=["customer_country", "customer_segment"], drop_first=True)
X_all = X_all.reindex(columns=X.columns, fill_value=0)

X_all_scaled = scaler.transform(X_all)

pred_pdf["predicted_buy"] = clf.predict(X_all_scaled)
pred_pdf["buy_probability"] = clf.predict_proba(X_all_scaled)[:, 1]

result_pdf = pred_pdf[
    [
        "customer_id",
        "customer_name",
        "customer_country",
        "customer_segment",
        "days_since_last_order_fixed",
        "total_orders",
        "total_revenue",
        "estimated_clv",
        "will_buy_again",
        "predicted_buy",
        "buy_probability"
    ]
]

result_spark = spark.createDataFrame(result_pdf)

result_spark.write.format("delta").mode("overwrite") \
    .saveAsTable("bi.gold.customer_purchase_predictions")

display(result_spark.orderBy("buy_probability", ascending=False))

In [0]:
df.groupBy("will_buy_again").count().show()

In [0]:
%sql
-- klienci z najwyższym prawdopodobieństwem zakupu
SELECT
    customer_name,
    customer_country,
    customer_segment,
    days_since_last_order_fixed AS days_since_last_order,
    total_orders,
    ROUND(total_revenue, 2) AS total_revenue,
    ROUND(estimated_clv, 2) AS estimated_clv,
    ROUND(buy_probability, 3) AS buy_probability
FROM bi.gold.customer_purchase_predictions
WHERE buy_probability >= 0.6
ORDER BY buy_probability DESC;

In [0]:
%sql
-- Macierz pomyłek
SELECT
    will_buy_again AS actual,
    predicted_buy AS predicted,
    COUNT(*) AS count
FROM bi.gold.customer_purchase_predictions
GROUP BY will_buy_again, predicted_buy
ORDER BY actual, predicted;

In [0]:
%sql
-- Średnie prawdopodobieństwo zakupu według segmentu klienta
SELECT
    customer_segment,
    COUNT(*) AS liczba_klientow,
    ROUND(AVG(buy_probability), 3) AS avg_buy_probability,
    ROUND(AVG(total_revenue), 2) AS avg_revenue,
    ROUND(AVG(estimated_clv), 2) AS avg_estimated_clv
FROM bi.gold.customer_purchase_predictions
GROUP BY customer_segment
ORDER BY avg_buy_probability DESC;